# LangSmith · Evaluating the Cookbook Assistant

`05_llamaindex` built an assistant that answers questions about this repo from the repo itself: a cited one-shot engine, and an agent that searches and opens files. Its checks were cheap and blunt: does the answer contain the right word, does it cite a right file. They passed an answer that put facts on the wrong side of a comparison.

This project measures that assistant properly, and uses the measurement to improve it. That loop is what LangSmith is for:

1. **Trace** every answer: each model call with its prompt, tokens and time; each tool call; what retrieval found.
2. **Evaluate** on a dataset: ten questions, each with what a right answer says and the repo's lines it rests on, scored by cheap checks and by a judge that checks the answer **claim by claim**.
3. **Fix** what the traces and verdicts point at, in 05's own code.
4. **Run again** as a new experiment, and compare the two in LangSmith.

There is no from-scratch build here: nothing in chapters 1–7 traces or evaluates. What this repo brings is the discipline the rest of it uses: everything offline against stand-ins first, the answer key checked against the files, and the judge checked against answers marked by hand before its numbers are trusted.

| File | What it is |
|---|---|
| `eval_set.py` | the ten questions: what a right answer says, each point resting on quoted words in the repo |
| `tracing.py` | where LangSmith goes into 05: its model clients, its tools, its retrieval |
| `claim_judge.py` | the judge: Gemini Flash, extracting claims first and checking them second |
| `cookbook_evals.py` | the dataset, experiments, answering, checks, judging, publishing, results |
| `eval_stubs.py` | a scripted Groq and a scripted Gemini, so all of it runs offline first |
| `traces_checks.py` | what building it found out about LangSmith, each reproduced |
| `test_evals.py` | offline tests for all of it |

---
# Part 1 · Build

## 1 · Where the tracer goes

LangSmith traces LangChain by itself, and LlamaIndex not at all. Its route for LlamaIndex is OpenTelemetry, which works for a run on its own and loses every LlamaIndex step inside an evaluation experiment (section 9 shows why). So this project traces at the seams 05 leaves open: the model clients, which every model call goes through, and the agent's tools. 05's `chat_model` takes a `wrap` for its raw clients, and the budget sits outside it:

```python
# ask_index.py
def chat_model(model=MODEL, budget=None, max_tokens=4096, wrap=None):
    """gpt-oss on Groq, through LlamaIndex's `OpenAILike`, with clients that count and cap what is spent.

    `OpenAILike` reuses the OpenAI clients it is handed, so the clients are where the budget lives:
    every call LlamaIndex makes -- an answer, an agent's turn -- goes through them. For the same reason
    they are where a tracer goes: `wrap`, if given, is applied to each raw client first (07_langsmith
    passes LangSmith's `wrap_openai`), and the budget sits outside it.
    """
    import openai
    budget = budget or Budget()
    key = os.environ["GROQ_API_KEY"]
    sync = openai.OpenAI(api_key=key, base_url=GROQ, max_retries=8)
    aio = openai.AsyncOpenAI(api_key=key, base_url=GROQ, max_retries=8)
    if wrap is not None:
        sync, aio = wrap(sync), wrap(aio)
    real_sync, real_aio = sync.chat.completions.create, aio.chat.completions.create

    def create(*args, **kwargs):
        for attempt in range(PARSE_RETRIES + 1):
            budget.check()
            try:
                response = real_sync(*args, **kwargs)
            except openai.BadRequestError as error:
                if attempt == PARSE_RETRIES or not _unparsed(error):
                    raise
                budget.retry(error)
                continue
            budget.charge(response)
            return response

    async def acreate(*args, **kwargs):
        for attempt in range(PARSE_RETRIES + 1):
            budget.check()
            try:
                response = await real_aio(*args, **kwargs)
            except openai.BadRequestError as error:
                if attempt == PARSE_RETRIES or not _unparsed(error):
                    raise
                budget.retry(error)
                continue
            budget.charge(response)
            return response

    sync.chat.completions.create, aio.chat.completions.create = create, acreate
    llm = _metered()(model=model, api_base=GROQ, api_key=key, openai_client=sync, async_openai_client=aio,
                     is_chat_model=True, is_function_calling_model=True, context_window=131_072,
                     max_tokens=max_tokens, temperature=0, timeout=120)
    llm._budget = budget
    return llm
```

What goes in at those seams. `traced_tool` exists because LangSmith's `traceable` adds a `config` keyword to a function's signature, and LlamaIndex reads a tool's schema off its signature: traced that way, the model would be offered an argument it was never offered untraced, and the traced system would not be the system being evaluated.

```python
# tracing.py
def wrap_client(client):
    """LangSmith's `wrap_openai`: every call on the client becomes an `llm` run."""
    from langsmith.wrappers import wrap_openai
    return wrap_openai(client)

def traced_tool(function):
    """A tool function as a LangSmith `tool` run, with the signature it had.

    `traceable` alone adds a `config` keyword to the function's signature, and LlamaIndex reads a tool's
    schema off the signature: the model would be offered a `config` argument it was never offered untraced,
    and a traced run would no longer be the system being evaluated. So the traced function is called through
    one whose signature is the original's.
    """
    from langsmith import traceable
    traced = traceable(run_type="tool", name=function.__name__)(function)

    @functools.wraps(function)
    def call(*args, **kwargs):
        return traced(*args, **kwargs)

    call.__signature__ = inspect.signature(function)
    return call

def traced_engine(query_engine):
    """The engine's retrieval as a `retriever` run: what was found, and how near, before any model call."""
    from langsmith import traceable
    retriever = query_engine.retriever
    inner = retriever._retrieve
    retriever._retrieve = traceable(run_type="retriever", name="CookbookRetriever",
                                    process_inputs=lambda inputs: {"query": inputs["query_bundle"].query_str},
                                    process_outputs=_passages)(inner)
    return query_engine
```

## 2 · The dataset

**Small on purpose: ten questions, each run once a round.** The size is set by the free tiers, not by what a good evaluation needs. Groq allows 200,000 tokens a day, and one round of these ten questions takes 60,000–100,000 of them (an agent question alone takes 10,000–45,000). The judge allows about 20 requests a day, which is ten answers. A real evaluation would first use many more questions: a difference averaged over a hundred questions means far more than one over ten. And because an agent's runs vary (here the same question passed one run and failed the next), it would also run each agent question a few times, which LangSmith does with `num_repetitions`, and report how consistently it succeeds. Every result here is one sample, and is read that way.

Seven one-shot questions, one per chapter 1–7, from 05's own set, for the engine; three that need two places in the repo, for the agent. Each has **reference points**, what a right answer says, and each point rests on **quotes**: words copied from the repo, which `unquoted()` checks are really in the evidence lines the judge is shown. A source that is later reworded breaks the check, rather than leaving a point that is no longer true. 82 quotes hold up 48 points. One of them, from the board question:

```python
("A board field can have one owner, the only agent allowed to write it: an agent gets a write tool only for "
 "the fields it owns, and a write to someone else's field is refused. A value two agents can both set is one "
 "neither can trust.",
 ["owned = [f for f, who in self.owner.items() if who == agent]",
  "raise Denied(f\"{field} is {owner}'s to set, not {author}'s\")",
  "a value two agents can both set is a value neither of them can trust."]),
```
Writing these points turned up wording that went past the source six times, the plainest: "skipped recall 7 of 15" where the README says it *consulted* memory 7 times in 15. And one place where the repo disagrees with itself: `state.py`'s docstring says a field's default writer is nobody, while `set()` lets anyone write a field without an owner. The point is written to what the code does.

```python
# eval_set.py
def evidence(question):
    """The evidence lines of one question, numbered, under the file they come from."""
    return "\n\n".join(f"[{rel}, lines {first}-{last}]\n" + "\n".join(f"{n:>4}  {line}" for n, line in
                                                                      _lines(rel, first, last))
                       for rel, first, last in question["evidence"])

def unquoted(question):
    """Every quote of a question's points that its evidence lines do not contain, as (point number, quote)."""
    shown = " ".join(_flat("\n".join(line for _, line in _lines(*span))) for span in question["evidence"])
    return [(i, quote) for i, (_, quotes) in enumerate(question["points"], 1) for quote in quotes
            if _flat(quote) not in shown]
```

## 3 · The experiment

An experiment is created empty before anything runs in it, so a run cut short can be finished in the same experiment. The target answers one question through 05, exactly as 05 runs, with the tracer in, and **saves the answer before returning it**: answers and verdicts spend different quotas, and nothing that happens to the judge may cost an answer again. A question cut off by Groq's daily limit is not an answer and is not saved, so the next run retries it.

```python
# cookbook_evals.py
def start(client, name, description="", metadata=None, dataset=DATASET):
    """An experiment, created empty if it does not exist yet, against the dataset."""
    if client.has_project(name):
        return client.read_project(project_name=name)
    return client.create_project(name, description=description, metadata=metadata or {},
                                 reference_dataset_id=client.read_dataset(dataset_name=dataset).id)

def target(index, name, chat_model=A.chat_model):
    """The system under test, as one async function: 05's engine for a one-shot question, 05's agent for a
    cross-chapter one, traced, metered, and saved before it returns."""
    from langsmith.run_helpers import get_current_run_tree

    async def answer(inputs):
        question, kind = inputs["question"], inputs["kind"]
        llm = chat_model(budget=A.Budget(BUDGETS[kind], f"07 {kind}"), wrap=T.wrap_client)
        began = time.monotonic()
        if kind == "one-shot":
            engine = T.traced_engine(A.engine(index, llm))
            try:
                # sync inside, so on a thread; the thread is given the current context, and the trace with it
                result = await asyncio.to_thread(A.ask, engine, question)
            except Exception as error:       # noqa: BLE001 -- a budget or a refused request is a stop, kept
                result = {"answer": None, "sources": [], "stopped": f"{type(error).__name__}: {error}"}
            output = {"answer": result["answer"], "stopped": result.get("stopped"), "calls": [],
                      "cited": sorted({s["where"].split(" § ")[0] for s in result["sources"]})}
        else:
            run = await A.run_agent(A.agent(index, llm, wrap=T.traced_tool), question)
            output = {"answer": run["answer"], "stopped": run["stopped"], "calls": run["calls"],
                      "cited": cited(run["answer"])}
        if output["stopped"] and "tokens per day" in output["stopped"]:
            raise GroqDayLimit(output["stopped"])
        run_tree = get_current_run_tree()
        output.update(kind=kind, usage=llm.budget.summary(), seconds=round(time.monotonic() - began, 1),
                      run=str(run_tree.id) if run_tree else None, trace=str(run_tree.trace_id) if run_tree else None)
        _save(name, question, output)
        return output

    return answer

async def run(client, index, name, dataset=DATASET, chat_model=A.chat_model, kinds=("one-shot", "agent")):
    """Answer what is missing, into the experiment, one question at a time (Groq's per-minute limit). `kinds` limits
    a round to the questions a change can affect."""
    from langsmith import aevaluate
    missing = to_answer(client, name, dataset, kinds)
    if not missing:
        return None
    return await aevaluate(target(index, name, chat_model=chat_model), data=missing, evaluators=CHECKS,
                           experiment=name, max_concurrency=1, client=client)
```

The free checks, scored by LangSmith as each answer arrives, are 05's: answered, cites a file the answer is in, contains the word a right answer must.

```python
# cookbook_evals.py
def answered(outputs, reference_outputs):
    return {"key": "answered", "score": int(bool(outputs.get("answer")) and not outputs.get("stopped"))}

def cites_an_answer_file(outputs, reference_outputs):
    return {"key": "cites an answer file", "score": int(bool(set(outputs.get("cited", [])) &
                                                              set(reference_outputs["files"])))}

def has_the_word(inputs, outputs, reference_outputs):
    if inputs["kind"] != "one-shot":
        return {"key": "has the word", "score": None}
    import cookbook as CB
    return {"key": "has the word", "score": int(CB.mentions(outputs.get("answer") or "", reference_outputs["word"]))}
```

## 4 · The judge

Gemini Flash, a different vendor from the gpt-oss model that writes the answers, so it is not grading its own family's writing, and on its own quota. It works in **two calls**:

1. **Extract**: shown the question and the answer, and nothing else, it lists every claim the answer makes, each with the answer's own words quoted.
2. **Check**: given those claims, the reference points and the evidence lines, it marks each claim *supported*, *contradicted* or *not in the sources*, and lists the points the answer makes.

It started as one call. Shown the key while it read the answer, it listed the **key**: it rewrote the answer's claims into the reference points' shape (it credited an answer with "in both, durability stops at the workflow", which the answer never said) and left out every claim the key had no point for, among them one that was wrong. Section 3 of Part 2 shows both versions on the same answers. Code checks what the model cannot be trusted to: that every quoted claim is in the answer, and every claim got a mark.

```python
# claim_judge.py
EXTRACT = """Below is an answer to a question about a code repository. List every claim it makes about the
repository: what something does, which part does it, a number, a measured result, a reason, a limitation, a file or
line it points to -- including claims inside tables, lists and code comments, and including claims about which file
a piece of code comes from. One claim per statement; a table cell or bullet that says two things is two claims.
Skip headings, restatements of the question, and general advice that says nothing about this repository.

For each claim, copy the answer's own words that make it, exactly as they appear.

QUESTION
{question}

ANSWER
{answer}"""
CHECK = """You check claims made in an answer about a code repository against the repository's own words.

QUESTION
{question}

REFERENCE POINTS -- what a right answer says. Each is true of the repo.
{points}

EVIDENCE -- the repo's lines the reference points come from, numbered.
{evidence}

THE ANSWER
{answer}

ITS CLAIMS, numbered, each with the answer's words that make it
{claims}

1. Mark every numbered claim, once:
   supported           the evidence or a reference point says it. Plausible is not enough: what neither of them
                       actually says is "not in the sources"
   contradicted       the evidence or a reference point says otherwise. This includes the right fact attached to the
                       wrong thing -- something one design does, credited to the other; a feature one has, said to be
                       missing from it; code from one file, said to be from another -- a wrong number, and the
                       reverse of what is said
   not in the sources  neither says it, nor says otherwise
   Read each claim as the answer makes it, in its place: a claim under a heading or in a column about one design
   is about that design, and in an answer comparing two things, "only X" means only X of those two.
2. List the reference points the answer makes. A point counts when the answer states its substance, in any words;
   not when it only mentions the topic.

Judge only against the evidence and the reference points, not against what you know of other software."""
```

```python
# claim_judge.py
def judge(question, answer, client=None, model=JUDGE_MODEL, sleep=time.sleep, log=print):
    """One answer's verdict: every claim with its quote and mark, the points made, and what judging cost."""
    if client is None and _groq(model):
        import openai
        client = openai.OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1",
                               max_retries=8)
    elif client is None:
        # its own key if there is one: Google stopped offering gemini-2.5-flash to new keys after this judge was
        # checked against the hand marks, and a different model would be a different judge, to be checked again
        from google import genai
        client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY_JUDGE") or os.environ["GEMINI_API_KEY"])
    extract, _ = prompts(question, answer)
    found, used = _ask(client, model, extract, Claims, sleep, log)
    claims = [c.model_dump() for c in found.claims]
    groups = _batches(question, answer, claims) if _groq(model) else [claims]
    marks, points, more, start = {}, None, 0, 0
    for group in groups:
        _, check = prompts(question, answer, group)
        marked, used = _ask(client, model, check, Marks, sleep, log)
        more += used
        marks.update({start + m.n: m for m in marked.marks if 1 <= m.n <= len(group)})
        # every batch sees the whole answer; the points it makes are read from the first
        points = marked.points_made if points is None else points
        start += len(group)
    for i, c in enumerate(claims, 1):
        m = marks.get(i)
        c.update(verdict=m.verdict if m else "unmarked", source=m.source if m else "", why=m.why if m else "")
    return {"claims": claims,
            "points made": sorted({p for p in (points or []) if 1 <= p <= len(question["points"])}),
            "check calls": len(groups),
            "unanchored": len(_unanchored(claims, answer)),
            "unmarked": sum(c["verdict"] == "unmarked" for c in claims),
            "judge": model, "judge tokens": used + more}

def scores(verdict, question):
    """A verdict as numbers. An answer that never came gets no scores: it is counted as unanswered instead."""
    if verdict.get("no answer"):
        return {}
    marks = [c["verdict"] for c in verdict["claims"]]
    return {"claims wrong": marks.count("contradicted"),
            "claims not in the sources": marks.count("not in the sources"),
            "claims": len(marks),
            "points made": round(len(verdict["points made"]) / len(question["points"]), 3)}
```

Gemini's free tier limits requests per minute and per day. A per-minute limit says how long to wait, and is waited out; a per-day one stops the run, and since verdicts are saved one at a time, the next run judges only what is missing. An overloaded model (503) is waited out too: the newest Flash answered "high demand" to every try for three minutes on the day this was built, which is why the judge is `gemini-3.5-flash`.

```python
# claim_judge.py
def _wait(error):
    """Seconds to wait before trying again, from a Gemini error; raises JudgeQuotaSpent for a per-day limit."""
    code = getattr(error, "code", None)
    body = getattr(error, "details", None) or {}
    details = body.get("error", {}).get("details", []) if isinstance(body, dict) else []
    if code == 429:
        quotas = [v.get("quotaId", "") for d in details for v in d.get("violations", [])]
        if any("PerDay" in q for q in quotas):
            raise JudgeQuotaSpent(f"{JUDGE_MODEL}: daily quota spent ({', '.join(quotas)})") from error
        delay = next((d["retryDelay"] for d in details if "retryDelay" in d), "30s")
        return float(re.sub(r"[^\d.]", "", delay) or 30) + 2
    if code in (500, 503, 504):
        return 30.0          # overloaded or unavailable: not a quota, try again shortly
    return None

def judge_all(answers, verdicts_path, questions=None, client=None, sleep=time.sleep, log=print):
    """Judge every saved answer that has no verdict yet, saving each verdict as it arrives.

    `answers` maps a question to its saved output ({"answer": ..., "stopped": ...}). Returns the verdicts, and
    whether it finished: False when the judge's daily quota ran out, and the rest waits for the next run.
    """
    questions = {q["question"]: q for q in (questions or E.QUESTIONS)}
    verdicts = json.loads(verdicts_path.read_text(encoding="utf-8")) if verdicts_path.exists() else {}
    for text, output in answers.items():
        if text in verdicts:
            continue
        if not output.get("answer"):
            # nothing to judge, and no point paying to be told so
            verdicts[text] = {"no answer": output.get("stopped") or "no answer"}
        else:
            try:
                verdicts[text] = judge(questions[text], output["answer"], client=client, sleep=sleep, log=log)
            except JudgeQuotaSpent as spent:
                log(f"    stopped: {spent}. {len(verdicts)} of {len(answers)} judged; run again after the reset.")
                return verdicts, False
        verdicts_path.parent.mkdir(parents=True, exist_ok=True)
        verdicts_path.write_text(json.dumps(verdicts, indent=1, ensure_ascii=False), encoding="utf-8")
        log(f"    judged {len(verdicts)} of {len(answers)}: {text[:60]}")
    return verdicts, True
```

## 5 · Publishing, and reading back

Verdicts go into the experiment as feedback, once every answer has one, through LangSmith's `evaluate()` on the existing experiment. And what LangSmith recorded is read back, so the notebook can show what each answer cost, step by step.

```python
# cookbook_evals.py
def publish(client, name, kinds=("one-shot", "agent"), log=print):
    """The verdicts into the experiment as feedback, once every answer of these kinds has one. Once per experiment."""
    from langsmith import evaluate
    verdicts = saved(name, "verdicts")
    answers = {q for q, a in saved(name).items() if a.get("kind", "one-shot") in kinds}
    if not answers <= set(verdicts):
        log(f"    {len(answers - set(verdicts))} answers have no verdict yet: run judge() first")
        return False
    marker = _path("verdicts", name).with_suffix(".published")
    if marker.exists():
        log("    already published")
        return True
    questions = {q["question"]: q for q in E.QUESTIONS}

    def verdict(inputs):
        if inputs["question"] not in verdicts:
            return {"results": []}                  # a kind not judged in this experiment
        found = verdicts[inputs["question"]]
        scored = J.scores(found, questions[inputs["question"]])
        wrong = [f"{c['claim']} -- {c['why']}" for c in found.get("claims", []) if c["verdict"] == "contradicted"]
        return {"results": [{"key": key, "score": value, **({"comment": "\n".join(wrong)}
                                                              if key == "claims wrong" and wrong else {})}
                            for key, value in scored.items() if key != "claims"]}

    evaluate(name, evaluators=[verdict], client=client)
    marker.write_text(time.strftime("%Y-%m-%d %H:%M"), encoding="utf-8")
    return True

def traces(client, name):
    """What LangSmith recorded for each question of an experiment, read back from it, and kept in store/traces/:
    every model call (tokens in and out, seconds), every tool call, and what retrieval returned."""
    runs = list(client.list_runs(project_name=name))
    found = []
    for root in sorted((r for r in runs if r.parent_run_id is None), key=lambda r: r.start_time):
        steps = sorted((r for r in runs if r.trace_id == root.trace_id and r.id != root.id), key=lambda r: r.start_time)
        seconds = lambda r: round((r.end_time - r.start_time).total_seconds(), 1) if r.end_time else None
        found.append({
            "question": root.inputs.get("question"), "seconds": seconds(root), "trace": str(root.trace_id),
            "model calls": [{"in": r.prompt_tokens, "out": r.completion_tokens, "seconds": seconds(r)}
                            for r in steps if r.run_type == "llm"],
            "tool calls": [{"tool": r.name, "args": r.inputs, "returned chars": len(str((r.outputs or {}).get("output", "")))}
                           for r in steps if r.run_type == "tool"],
            "retrieved": [d["metadata"]["file"] for r in steps if r.run_type == "retriever"
                          for d in (r.outputs or {}).get("documents", [])]})
    _path("traces", name).parent.mkdir(parents=True, exist_ok=True)
    _path("traces", name).write_text(json.dumps(found, indent=1, ensure_ascii=False), encoding="utf-8")
    return found
```

---
# Part 2 · Run and show

In [1]:
import asyncio
import json
import warnings

warnings.filterwarnings("ignore")

import eval_set as E
import ask_index as A
import claim_judge as J
import cookbook_evals as C
import traces_checks as K
from langsmith import Client


def show(value):
    print(json.dumps(value, indent=1, ensure_ascii=False, default=str))


client = Client()
BASE, FIX, TRIM = "cookbook-baseline", "cookbook-fix-1", "cookbook-fix-2"     # the experiments, as named in LangSmith

## 1 · Offline

The whole pipeline against stand-ins: 05's real engine and agent over a small index, a scripted Groq at the network layer, LangSmith's wrappers on. First the check that matters most, that tracing changes nothing the model is sent:

In [2]:
import test_evals as TE

small = TE.small_index()
A.FLOOR = 0.0          # the stand-in embedding's scores are not real similarities; restored below
show(await K.tracing_is_invisible(small, "What does compaction leave untouched?", E.QUESTIONS[7]["question"]))
A.FLOOR = 0.66

{
 "requests, untraced": 3,
 "requests, traced": 3,
 "identical": true,
 "tools offered, traced": [
  "search_cookbook['query']",
  "open_file['line_end', 'line_start', 'path']"
 ]
}


## 2 · The dataset, in LangSmith

Created once; an example whose points change is updated, and LangSmith keeps the version each experiment ran against.

In [3]:
dataset, change = C.ensure_dataset(client)
print(dataset.name, change)
for q in E.QUESTIONS:
    print(f"  {q['kind']:9} {len(q['points'])} points, {sum(len(x) for _, x in q['points']):2} quotes   {q['question'][:70]}")
print("quotes not found in their evidence:", sum(len(E.unquoted(q)) for q in E.QUESTIONS))

ask-the-cookbook {'added': 0, 'updated': 0, 'unchanged': 10}
  one-shot  5 points,  8 quotes   The README says extract() asks the API to enforce the schema first. If
  one-shot  5 points,  6 quotes   What counts as a doom loop, and what does the runtime do when it detec
  one-shot  4 points,  6 quotes   What is the difference between a tool that refuses and one that is bro
  one-shot  4 points,  6 quotes   Why is recall injected into the loop while remembering is a tool the a
  one-shot  4 points,  5 quotes   What does compaction leave untouched, and why?
  one-shot  4 points,  7 quotes   What is the difference between handing off a conversation and delegati
  one-shot  4 points,  9 quotes   Does running an MCP server as a separate process sandbox it? Could it 
  agent     6 points, 16 quotes   How is chapter 6's shared board different from chapter 4's memory stor
  agent     7 points, 12 quotes   In chapter 8's LangGraph project, compare how the LangGraph desk and t
  agent     7 poin

## 3 · A judge is test equipment: it is checked before it is trusted

An LLM judge is a model asked to grade another model's work. Before its grades mean anything, it gets the treatment any measuring instrument gets: compare it against something known. The usual procedure:

1. **A labelled set.** People grade a set of real answers by hand, with the same questions the judge will be asked: here, which claims are wrong, and which reference points the answer makes. Calls that could go either way are recorded as such, not forced.
2. **A split.** One part is used to build and tune the judge (its prompt, its model). The other part is **held out**, never looked at while tuning, and used only to measure.
3. **Agreement, measured on the held-out part,** for each thing the judge reports separately. For wrong claims: of the errors people found, how many it caught, and how many of its alarms were errors at all. For points made: how often its list matches.
4. **A decision.** Use the judge for what it agrees on; don't use it for what it doesn't; or improve it, and measure again on answers it was not tuned on.
5. **Again, whenever the judge changes.** A new model or a new prompt is a new instrument.

Here: 2 answers from 05 to design it on (this section), the baseline's answers to tune it on (section 5), and 05's own saved answers, 10 of them, held out.

**Designing it.** Two answers from 05, marked by hand before the judge saw them: the LlamaIndex agent on the board question (nothing wrong), and the scratch agent on the LangGraph question (three claims wrong). The judge in one call, and in two. Saved results; no model call here.

In [3]:
check = json.load(open("store/judge_check.json", encoding="utf-8"))
qs = {"llamaindex board": E.QUESTIONS[7], "scratch langgraph": E.QUESTIONS[8]}
print(f"{'':20}{'by hand':>10}{'one call':>22}{'two calls':>22}")
for name, q in qs.items():
    hand = check["checked by hand"][name]["claims wrong"]
    row = [f"{J.scores(check[k][name], q)['claims wrong']} wrong of {len(check[k][name]['claims'])}"
           for k in ("one call (judge shown the key while reading the answer)", "two calls (claims first, then the key)")]
    print(f"{name:20}{hand:>10}{row[0]:>22}{row[1]:>22}")
print()
for k in ("one call (judge shown the key while reading the answer)", "two calls (claims first, then the key)"):
    print(k)
    for c in check[k]["scratch langgraph"]["claims"]:
        if c["verdict"] == "contradicted":
            print("   wrong:", c["claim"][:160])

                       by hand              one call             two calls
llamaindex board             0          0 wrong of 7         0 wrong of 27
scratch langgraph            3         2 wrong of 11         4 wrong of 38

one call (judge shown the key while reading the answer)
   wrong: In LangGraph, the node returns a Wait object that tells the system which field to wait on.
   wrong: The code excerpt starting with `after = self.steps[case.values["next"]](case)` is from `ap_graph.py`.
two calls (claims first, then the key)
   wrong: In LangGraph desk, the node returns a Wait object that tells the system which field to wait on
   wrong: In LangGraph desk, parallelism, streaming, rewinding/forking, timeout, or API serving are not provided out-of-the-box
   wrong: In ap_graph.py, the desk sets the waiting field to the asked value
   wrong: In ap_graph.py, the desk clears the next field


In one call the judge found 11 claims in an answer of about 40, and missed a wrong one: LangGraph "lacks" parallelism and streaming, which the evidence table says it has. It had rewritten the answer's claims into the shape of the reference points, and left out what the key had no point for. In two calls (claims first, without the key; then each claim checked) it caught all three mistakes. The mislabelled code is two claims (it sets the waiting field, it clears the next one), so four. On the board answer both found nothing wrong, as by hand. The two-call design is kept. Two answers prove little; the real measurement is section 5.

## 4 · Baseline, live

05 as it is, all ten questions, one at a time. Each answer is saved as it arrives.

In [2]:
index = A.open_index()
BASE = "cookbook-baseline"
C.start(client, BASE, description="05_llamaindex as built", metadata={"system": "05 as built", "model": A.MODEL})
await C.run(client, index, BASE)
for r in C.results(BASE):
    print(f"{r['kind']:9} answered={r['answered']!s:5} cites={r['cites an answer file']!s:5} word={r['has the word']!s:5}"
          f" calls={r['tool calls']} tokens={r['tokens']:>6,} largest request={r['largest request']:>5,}  {r['question'][:50]}")
    if r["stopped"]:
        print("          stopped:", r["stopped"][:160])

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


View the evaluation results for experiment: 'cookbook-baseline' at:
https://smith.langchain.com/o/0fdf5d9d-7d7b-4da3-9ce3-ece5ff4b3dde/datasets/6d079e56-9d23-4f6b-acbf-c5b786c833cc/compare?selectedSessions=1fab8080-589c-45a9-bf93-8132ac26c169




0it [00:00, ?it/s]

1it [00:06,  6.95s/it]

2it [00:36, 20.47s/it]

...

8it [05:44, 34.08s/it]

9it [07:20, 53.30s/it]

10it [07:50, 46.15s/it]

10it [07:51, 47.12s/it]

one-shot  answered=True  cites=True  word=True  calls=0 tokens= 2,621 largest request=2,178  The README says extract() asks the API to enforce 
one-shot  answered=True  cites=True  word=True  calls=0 tokens= 3,741 largest request=3,072  What counts as a doom loop, and what does the runt
one-shot  answered=True  cites=True  word=True  calls=0 tokens= 2,849 largest request=2,398  What is the difference between a tool that refuses
one-shot  answered=True  cites=True  word=True  calls=0 tokens= 3,718 largest request=3,288  Why is recall injected into the loop while remembe
one-shot  answered=True  cites=True  word=True  calls=0 tokens= 3,630 largest request=3,119  What does compaction leave untouched, and why?
one-shot  answered=True  cites=True  word=True  calls=0 tokens= 3,610 largest request=3,050  What is the difference between handing off a conve
one-shot  answered=True  cites=True  word=True  calls=0 tokens= 3,010 largest request=2,693  Does running an MCP server as a separate proces

**What the baseline showed**, on the free checks alone, for about 63,000 Groq tokens:

- **All seven one-shot answers pass**: answered, citing a file the answer is in, with the word a right answer needs.
- **The board question was answered, and cited nothing usable.** It cites what it read as `【open_file†L1-L7】`, the tool's name with line numbers, and never the file. The answer itself is right, by hand.
- **The LangGraph question got no answer.** Six tool calls went well; on the seventh, gpt-oss wrote its reasoning where a tool call should be, Groq refused it (`400 output_parse_failed`), and LlamaIndex's agent has no way to carry on from a refused call. Six calls' work, lost.
- **The third agent question passed.**

Every answer was then marked by hand (`hand_labels.py`), before any judge saw it: 1 claim wrong in the nine, and 2 borderline.

## 5 · Validating the judge, and where it got stuck

**Tuning, on the baseline's answers.** The first judge model, `gemini-3.5-flash`, judged 8 of the baseline's answers before its daily limit. Compared with the hand marks it showed two habits, and the check step was tightened against both: it marked "supported" things the sources never say ("plausible is not enough" was added), and it read "only the refused outcome is passed back" outside the comparison the answer was making. Those 8 verdicts are kept (`store/verdicts/cookbook-baseline.gemini-3.5-flash.partial.json`) and not used as scores, so no experiment is graded by two judges.

One hand mark was changed after the judge ran, and says so in `hand_labels.py`: the baseline's doom-loop answer, first marked wrong, then moved to borderline under the rule set before judging (an omission is not a false statement). Discount it accordingly.

**The bind: the models a judge could be.** Every candidate was tried on the held-out answers, the same 20 requests each:

In [2]:
import hand_labels as H

for t in H.JUDGE_TRIALS:
    if not t["ran"]:
        print(f"{t['judge']:28} could not run: {t['why not']}")
        continue
    print(f"{t['judge']:28} wrong claims caught: {len(t['caught'])} of {len(t['caught']) + len(t['missed'])};"
          f" false alarms: {t['false alarms'].split(':')[0]}")
    print(f"{'':28} points made: {t['points']}")

gemini-3.8-flash             could not run: 503 'high demand' on every try, twice, an hour apart
qwen/qwen3.8-27b (Groq)      could not run: judged 3 of the 10 answers, then was refused on its limit of 1,000 output tokens a minute; a claim list is about that long, so a round would be mostly waiting
gemini-2.5-flash             wrong claims caught: 2 of 5; false alarms: 7
                             points made: 7 of 10 answers exactly as by hand; the other 3 differ by one borderline point
openai/gpt-oss-20b (Groq)    wrong claims caught: 0 of 5; false alarms: 1
                             points made: 4 of 10 exactly as by hand; on the compaction answer it found no claims at all; on the scratch board answer it found 3 of 6 points


The judge has to be a different model from the one it grades, and this account's free tiers leave two kinds:

- **Big models, rationed.** The Gemini Flash models allow about **20 requests a day each** on the free tier. The judge makes two an answer, so one round of ten answers is one day's allowance, and checking a judge is another. The newest, `gemini-3.8-flash`, could not be reached at all ("503, high demand", twice, an hour apart). `qwen3.8-27b` on Groq judged 3 of the 10 answers, then was refused on its limit of 1,000 output tokens a minute; a list of claims is about that long, so a round would be mostly waiting.
- **A small model, plentiful and weak.** `gpt-oss-20b` on Groq has no daily request cap that matters here. It caught none of the five errors, and on one answer it found no claims at all.

`gemini-2.5-flash` is the judge. Its record on the held-out answers, against the hand marks:

In [3]:
for title, labels, verdicts in [
        ("baseline (tuned on)", H.BASELINE, C.saved(BASE, "verdicts")),
        ("held out", H.HELD_OUT, json.load(open(f"store/verdicts/held-out.{J.JUDGE_MODEL}.json", encoding="utf-8")))]:
    rows = H.side_by_side(labels, verdicts)
    print(f"{title}: {len(rows)} answers")
    print(f"   wrong claims, by hand: {sum(r['wrong, by hand'] for r in rows)} (+{sum(r['borderline, by hand'] for r in rows)} borderline)"
          f"   by the judge: {sum(r['wrong, by the judge'] for r in rows)}")
    for r in rows:
        who = r["answer"] if isinstance(r["answer"], str) else f"{r['answer'][0]}: {r['answer'][1]}"
        print(f"   hand {r['wrong, by hand']}+{r['borderline, by hand']}b  judge {r['wrong, by the judge']}"
              f"   points hand {r['points, by hand']} judge {r['points, by the judge']}   {who[:60]}")
        for w in r["judge's wrong claims"]:
            print("        judge:", w[:150])
    print()

baseline (tuned on): 2 answers
   wrong claims, by hand: 0 (+0 borderline)   by the judge: 3
   hand 0+0b  judge 2   points hand [1, 2, 3, 4, 5, 6] judge [1, 2, 3, 4, 5, 6]   How is chapter 6's shared board different from chapter 4's m
        judge: Agents write to the board with a tool that checks ownership and read-access. -- The write tool checks ownership, not read-access. Read access is deter
        judge: The board’s read/write is tool-driven. -- Reading the board is described as injection, not tool-driven. Writing is tool-driven.
   hand 0+0b  judge 1   points hand [1, 2, 3, 4] judge [1, 2, 3, 4]   How does chapter 2's runtime stop an agent that keeps callin
        judge: Chapter 2's runtime triggers a stop upon detection of a second identical tool call or other loop patterns. -- The evidence only supports detection of 

held out: 10 answers
   wrong claims, by hand: 5 (+3 borderline)   by the judge: 13
   hand 0+0b  judge 0   points hand [1, 2, 4, 5] judge [1, 2, 4, 5]   eng

**The decision.**

- **Points made: used.** Its list matches the hand marks exactly on 7 of the 10 held-out answers, and is one borderline point off on the other 3. That is good enough to compare rounds.
- **Wrong claims: not used as a score.** It caught 2 of the 5 errors people found, and raised 7 false alarms. Six of those were the engine's citation markers (`【4†L1-L5】`: source 4, lines 1–5 of that passage) read as claims about file line numbers; the seventh was the "only" reading again. A count like that says nothing about whether a round got better. Its list of contradicted claims is still useful, as leads for a person to check.
- **Not tuned further.** The citation-marker false alarms have an obvious fix, but they were found on the held-out answers. Fixing them there would leave nothing unseen to measure the fixed judge on; the next step would be a fresh held-out set.

What would get unstuck, in order of cost: a larger hand-marked set (more errors to measure catching on: five is few); a paid key, which takes the 20-a-day limit away from the stronger models; or a judge that checks one claim per call, which suits a small model better than a list of forty.

## 6 · What the traces show

Read back from LangSmith: every model call each answer made, tokens in and out, and what retrieval and the tools returned.

In [3]:
for t in C.traces(client, BASE):
    calls = t["model calls"]
    print(f"{t['question'][:60]:60} {len(calls)} model calls, {t['seconds']}s")
    print("      tokens in per call:", [c["in"] for c in calls])
    for tool in t["tool calls"]:
        print(f"      {tool['tool']}({tool['args']})  -> {tool['returned chars']:,} chars")

What is the difference between a tool that refuses and one t 1 model calls, 6.9s
      tokens in per call: [2398]
How does chapter 2's runtime stop an agent that keeps callin 4 model calls, 29.9s
      tokens in per call: [435, 1375, 2364, 3800]
      search_cookbook({'query': 'chapter 2 runtime stop agent keeps calling same tool'})  -> 3,744 chars
      search_cookbook({'query': 'LangGraph limit chapter 8'})  -> 3,883 chars
      open_file({'line_end': 230, 'line_start': 150, 'path': '02_agent_runtime/runtime.py'})  -> 5,826 chars
In chapter 8's LangGraph project, compare how the LangGraph  7 model calls, 188.2s
      tokens in per call: [436, 1142, 2106, 2845, 4466, 5388, 0]
      search_cookbook({'query': 'chapter 8 LangGraph desk wait for a person'})  -> 2,509 chars
      search_cookbook({'query': 'LangGraph desk waiting person case file wait for a person'})  -> 3,639 chars
      search_cookbook({'query': 'LangGraph desk waiting'})  -> 2,886 chars
      open_file({'line_end': 460, 

**Read from the traces:**

- **The LangGraph question:** six calls whose requests grow 436 → 5,388 tokens, then a seventh with no tokens at all. That is the call Groq refused, after 188 seconds in all.
- **The board question:** two searches, then two `open_file` reads of about 5,700 characters each. Those reads are what it cited as `open_file`.
- **The one-shot answers:** one model call each, of 2,200–3,300 tokens in.

A trace says what happened, not why the model wrote what it wrote. But each fix below starts from one.

## 7 · Fixes 1 and 2, then round 2

Both come from the agent's traces, and both are measured by the free checks, so neither depends on the judge.

**Fix 1: the agent cited a tool, not a file.** On the board question it read `state.py` and `stores.py` through `open_file`, and cited them as `【open_file†L1-L7】`: `open_file` returned numbered lines with nothing saying which file they were. Search results are labelled with their file and lines; now `open_file`'s are too.

```python
# ask_index.py
def open_file(path: str, line_start: int = 1, line_end: int = FILE_LINES) -> str:
    """Read lines of one cookbook file, by the path a search result gave, at most 100 lines at a time.
    Notebooks are read as their prose."""
    if path not in readable:
        # only what is indexed: not .env, not anything outside the repo, whatever the path says
        return f"Not a cookbook file: {path!r}. Use a path exactly as search_cookbook labels it."
    repeated = again("open", path, line_start, line_end)
    if repeated:
        return repeated
    file = readable[path]
    text = CB._notebook_prose(file) if file.suffix == ".ipynb" else file.read_text(encoding="utf-8")
    lines = text.splitlines()
    start = max(1, line_start)
    end = min(len(lines), line_end, start + MAX_FILE_LINES - 1)
    # labelled as a search result is: without it, the agent cited what it read as "【open_file†L1-L7】" -- the
    # tool's name, never the file (07_langsmith's baseline, on the board question)
    return f"[{path}, lines {start}-{end}]\n" + \
        "\n".join(f"{i:>4}  {lines[i - 1]}" for i in range(start, end + 1)) + \
        (f"\n... ({len(lines)} lines in all)" if end < len(lines) else "")
```

**Fix 2: one unreadable reply lost a whole answer.** On the LangGraph question, after six good calls, gpt-oss wrote its reasoning where a tool call should be, Groq refused it (`400 output_parse_failed`), and LlamaIndex's agent has no way to carry on from a refused call. The same request usually succeeds, so 05's metered client now sends it again, up to twice, counted; any other 400 still raises. The client is the right place: every call goes through it, and so does the budget.

```python
# ask_index.py
# Groq refuses a reply it cannot parse as a tool call or an answer: 400 `output_parse_failed`, when gpt-oss writes
# its reasoning where the call should be, and `tool_use_failed` (02_langgraph met it). It is the model's slip, not
# the request's fault, and the same request usually succeeds -- but LlamaIndex's agent has no way to recover from
# it, and lost a whole answer to it after six good calls (07_langsmith's baseline, on the LangGraph question). So
# such a request is sent again, up to twice, and counted. Any other 400 is a real error, and raises.
PARSE_RETRIES = 2
```

```python
# ask_index.py
def _unparsed(error):
    return getattr(error, "code", None) in _PARSE_FAILURES or any(c in str(error) for c in _PARSE_FAILURES)
```

**Round 2:** 05 with both fixes, the same ten questions, a new experiment.

In [2]:
index = A.open_index()
C.start(client, FIX, description="05 with open_file naming its file, and unparsed replies sent again",
        metadata={"system": "05 + open_file label + parse retry", "model": A.MODEL})
await C.run(client, index, FIX)
for r in C.results(FIX):
    print(f"{r['kind']:9} answered={r['answered']!s:5} cites={r['cites an answer file']!s:5} word={r['has the word']!s:5}"
          f" calls={r['tool calls']} tokens={r['tokens']:>6,} largest request={r['largest request']:>5,}  {r['question'][:50]}")
    if r["stopped"]:
        print("          stopped:", r["stopped"][:160])

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


View the evaluation results for experiment: 'cookbook-fix-1' at:
https://smith.langchain.com/o/0fdf5d9d-7d7b-4da3-9ce3-ece5ff4b3dde/datasets/6d079e56-9d23-4f6b-acbf-c5b786c833cc/compare?selectedSessions=6d396e8f-bfa4-4175-8c12-7cb31967b259




0it [00:00, ?it/s]

1it [00:06,  6.83s/it]

2it [00:12,  6.01s/it]

...

8it [05:20, 33.43s/it]

9it [06:53, 52.10s/it]

10it [07:21, 44.62s/it]

10it [07:22, 44.25s/it]

one-shot  answered=True  cites=True  word=True  calls=0 tokens= 2,711 largest request=2,178  The README says extract() asks the API to enforce 
one-shot  answered=True  cites=True  word=True  calls=0 tokens= 3,670 largest request=3,072  What counts as a doom loop, and what does the runt
one-shot  answered=True  cites=True  word=True  calls=0 tokens= 2,849 largest request=2,398  What is the difference between a tool that refuses
one-shot  answered=True  cites=True  word=True  calls=0 tokens= 3,716 largest request=3,288  Why is recall injected into the loop while remembe
one-shot  answered=True  cites=True  word=True  calls=0 tokens= 3,646 largest request=3,119  What does compaction leave untouched, and why?
one-shot  answered=True  cites=True  word=True  calls=0 tokens= 3,624 largest request=3,050  What is the difference between handing off a conve
one-shot  answered=True  cites=True  word=True  calls=0 tokens= 3,090 largest request=2,693  Does running an MCP server as a separate proces

The agent questions' traces in round 2:

In [2]:
for t in C.traces(client, FIX):
    if t["tool calls"]:
        print(f"{t['question'][:60]:60} {len(t['model calls'])} model calls, {t['seconds']}s")
        print("      tokens in per call:", [c["in"] for c in t["model calls"]])

How does chapter 2's runtime stop an agent that keeps callin 3 model calls, 5.4s
      tokens in per call: [435, 1375, 2364]
In chapter 8's LangGraph project, compare how the LangGraph  9 model calls, 187.4s
      tokens in per call: [436, 1142, 1673, 2663, 3623, 4545, 5842, 6584, 0]
How is chapter 6's shared board different from chapter 4's m 5 model calls, 93.2s
      tokens in per call: [424, 1313, 1958, 3452, 4962]


**What round 2 showed**, one run, so one sample of each:

- **Fix 1 worked: the board question now cites its files.**
- **Fix 2 was not tested.** No reply was refused this time.
- **The LangGraph question still got no answer, for a different reason.** The agent searched and read for eight tool calls, and its requests grew with each: 436, 1,142, 1,673, 2,663, 3,623, 4,545, 5,842, 6,584 tokens. The next one was over Groq's limit of 8,000 (`413`). That is the failure 05 found first: LlamaIndex's agent sends everything it has found on every call, and its memory limit does not trim within a run.
- **Everything else answered as before,** at about the same cost.

## 8 · Fixes 3 and 4, then round 3

**Fix 3: each request cut to fit.** Every request an agent makes passes through 05's metered model (`achat`): its tool steps, and the answer it is made to give at its step limit. So that is where the cut goes. Above 6,000 tokens of messages, the oldest tool results are cut to their labels (file, section, lines), oldest first, until the request fits. The question, the system prompt, every tool call, the two latest results and every call id stay whole. The run keeps everything; each request gets a part chosen to fit. That is chapter 5's idea, in LlamaIndex's terms.

```python
# ask_index.py
def fit(messages, limit):
    """What one request is sent: every message, or -- when that is over `limit` tokens -- every message with the oldest
    tool results cut to their labels, oldest first, until it fits. The question, the system prompt, every tool call
    and the latest KEEP_WHOLE results are never cut; each cut result keeps its tool_call_id, so the request stays
    one the API accepts. The conversation itself is not changed: only what this request carries.

    Chapter 5's lesson in LlamaIndex terms: the run keeps everything; each call gets a part of it chosen to fit.
    """
    from llama_index.core.base.llms.types import ChatMessage, MessageRole
    if limit is None:
        return messages
    sizes = [_tokens(m) for m in messages]
    total = sum(sizes)
    if total <= limit:
        return messages
    sent = list(messages)
    older = [i for i, m in enumerate(messages) if m.role == MessageRole.TOOL][:-KEEP_WHOLE]
    for i in older:
        if total <= limit:
            break
        cut = ChatMessage(role=MessageRole.TOOL, content=_labels(messages[i].content),
                          additional_kwargs=dict(messages[i].additional_kwargs))
        total -= sizes[i] - _tokens(cut)
        sent[i] = cut
    return sent
```

**Fix 4, found while testing fix 3: the step-limit answer was written blind.** Against a scripted Groq that keeps searching, the last request, the one where the agent must answer after eight tool calls, carried three messages: the system prompt, the question and LlamaIndex's stop notice. None of the eight results was in it. LlamaIndex builds that request from the agent's memory, and during a run the tool calls and results live in a step "scratchpad", moved into memory only when a run finishes normally. No live run had reached the limit yet; with fix 3 one would, and its answer would have been written from nothing. 05's agent now moves the scratchpad into memory first, as LlamaIndex's own `finalize` does, leaving out a last call that never got its result.

The code below is this fix as it stands after `08_langfuse`. There, gpt-oss answered the step-limit request with yet another tool call, which Groq refuses, on every try: a request that is all tool calls invites one more. So the findings now go in as plain text, in one message. Round 3 here ran the first version.

```python
# ask_index.py
def _finishing():
    """LlamaIndex's `FunctionAgent`, with its step-limit answer given what the run found.

    `early_stopping_method="generate"` makes one last request when `max_iterations` is reached, with no tools, for an
    answer. LlamaIndex builds it from the agent's memory -- and during a run, the tool calls and results are not in
    memory but in the step "scratchpad", moved into memory only when the run finishes normally (`finalize`). So the
    last request carried the system prompt, the question and the stop notice, and nothing the eight tool calls had
    found: an answer written blind (test_ask.test_no_agent_request_outgrows_its_limit, which first showed it).

    So what the run found goes into memory first -- as one message of plain text (`_found`), not as the run's tool
    calls and results. The first version of this fix moved the tool messages in as they were, and on 08_langfuse's
    LangGraph question (prompt v2) gpt-oss answered the no-tools request with yet another tool call, which Groq
    refuses, on every try -- including two after NO_TOOLS_NOW was added: a request that is all tool calls invites
    one more. With the results as text there is no call in the request to continue. If the model calls a tool
    anyway, it is asked once more with NO_TOOLS_NOW said plainly.
    """
    global _FINISHING
    if _FINISHING is None:
        from llama_index.core.agent.workflow import FunctionAgent

        class FinishingAgent(FunctionAgent):
            async def _generate_early_stopping_response(self, ctx, max_iterations):
                import openai
                from llama_index.core.base.llms.types import ChatMessage
                scratchpad = await ctx.store.get(self.scratchpad_key, default=[])
                memory = await ctx.store.get("memory")
                await memory.aput(ChatMessage(role="user", content=_found(scratchpad)))
                await ctx.store.set(self.scratchpad_key, [])
                try:
                    return await super()._generate_early_stopping_response(ctx, max_iterations)
                except openai.BadRequestError as error:
                    if not _unparsed(error) and "Tool choice is none" not in str(error):
                        raise
                    await memory.aput(ChatMessage(role="user", content=NO_TOOLS_NOW))
                    return await super()._generate_early_stopping_response(ctx, max_iterations)

        _FINISHING = FinishingAgent
    return _FINISHING
```

**Round 3** runs only the three agent questions: both changes touch only the agent, and the one-shot path is the same as in round 2.

In [2]:
index = A.open_index()
C.start(client, TRIM, description="round 2 + the agent's requests cut to fit, and its step-limit answer given the run",
        metadata={"system": "05 + label + parse retry + fit + finishing", "model": A.MODEL})
await C.run(client, index, TRIM, kinds=("agent",))
for r in C.results(TRIM):
    print(f"answered={r['answered']!s:5} cites={r['cites an answer file']!s:5} calls={r['tool calls']}"
          f" tokens={r['tokens']:>6,} largest request={r['largest request']:>5,}  {r['question'][:60]}")
    if r["stopped"]:
        print("          stopped:", r["stopped"][:160])
for t in C.traces(client, TRIM):
    print(f"{t['question'][:60]:60} tokens in per call: {[c['in'] for c in t['model calls']]}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


View the evaluation results for experiment: 'cookbook-fix-2' at:
https://smith.langchain.com/o/0fdf5d9d-7d7b-4da3-9ce3-ece5ff4b3dde/datasets/6d079e56-9d23-4f6b-acbf-c5b786c833cc/compare?selectedSessions=182642b9-a7af-4f89-90df-14aaa332a2f9




0it [00:00, ?it/s]

1it [00:37, 37.85s/it]

2it [05:35, 190.64s/it]

3it [07:59, 169.57s/it]

3it [08:01, 160.39s/it]

answered=True  cites=True  calls=5 tokens=19,744 largest request=5,777  How is chapter 6's shared board different from chapter 4's m
answered=True  cites=True  calls=8 tokens=42,675 largest request=6,215  In chapter 8's LangGraph project, compare how the LangGraph 
answered=True  cites=True  calls=4 tokens=12,841 largest request=3,910  How does chapter 2's runtime stop an agent that keeps callin


How does chapter 2's runtime stop an agent that keeps callin tokens in per call: [435, 1375, 2364, 3287, 3910]
In chapter 8's LangGraph project, compare how the LangGraph  tokens in per call: [436, 1142, 2106, 2845, 4142, 5108, 6215, 5836, 6207, 6043]
How is chapter 6's shared board different from chapter 4's m tokens in per call: [424, 1313, 1958, 3468, 4962, 5777]


**What round 3 showed**, one run:

- **The LangGraph question was answered, for the first time in any round, and answered well.** Its requests level off near 6,000 tokens instead of growing until one is refused: 5,108, 6,215, 5,836, 6,207, 6,043. After eight tool calls it gave the step-limit answer, built from what the run found (fix 4 at work).
- **By hand:** no claim wrong. It explains `interrupt()` and the re-run node, and the `Wait` that ends a step. It gives all three LangGraph hazards and the case file's answer to each, and quotes the right code under the right file names (its `interrupt(...)` line is `ap_graph.py` line 292, shown in the notebook at prose line 97, inside the lines it cites). One overclaim, borderline: the case file's "only failure mode" is a `Denied`, where the notebook says durability stops at the workflow in both. It makes 5 of the 7 points.
- **The other two agent questions answered and cited, as in round 2.**
- **It is the most expensive answer in any round: 42,675 tokens,** for eight full tool calls and the answer after them. The cut caps each request, not the run. The run's budget does that, and 45,000 tokens a question let it through.

## 9 · The three rounds, side by side

The free checks and the cost, from the saved results. The same comparison is in LangSmith, on the dataset's page, with each answer's trace a click away.

In [2]:
rounds = [("baseline", BASE), ("round 2", FIX), ("round 3", TRIM)]
found = {label: {r["question"]: r for r in C.results(name)} for label, name in rounds}
for q in E.QUESTIONS:
    print(q["question"][:90])
    for label, _ in rounds:
        r = found[label].get(q["question"])
        if r is None:
            print(f"   {label:9} (not run: the change did not touch this question)")
            continue
        print(f"   {label:9} answered={r['answered']!s:5} cites={r['cites an answer file']!s:5} tool calls={r['tool calls']}"
              f"  tokens={r['tokens']:>6,}  largest request={r['largest request']:>5,}"
              + (f"  stopped: {r['stopped'][:40]}" if r["stopped"] else ""))

The README says extract() asks the API to enforce the schema first. If the API enforces it
   baseline  answered=True  cites=True  tool calls=0  tokens= 2,621  largest request=2,178
   round 2   answered=True  cites=True  tool calls=0  tokens= 2,711  largest request=2,178
   round 3   (not run: the change did not touch this question)
What counts as a doom loop, and what does the runtime do when it detects one?
   baseline  answered=True  cites=True  tool calls=0  tokens= 3,741  largest request=3,072
   round 2   answered=True  cites=True  tool calls=0  tokens= 3,670  largest request=3,072
   round 3   (not run: the change did not touch this question)
What is the difference between a tool that refuses and one that is broken, and which does 
   baseline  answered=True  cites=True  tool calls=0  tokens= 2,849  largest request=2,398
   round 2   answered=True  cites=True  tool calls=0  tokens= 2,849  largest request=2,398
   round 3   (not run: the change did not touch this question)
Why i

**Comparing designs, not only prompts.** LangSmith compares experiments, and an experiment is the outputs, scores and traces of whatever function ran over the dataset. Its comparison works the same whether the change between two experiments was a prompt or the architecture: a request cut, a new step at the step limit. But LangSmith keeps the results, not the code. Once a fix is made, the design it replaced is gone from the code, unless it was kept somewhere. Three habits keep it, and they need nothing more than git:

1. **Commit before each experiment.** The code that ran is then exactly one commit.
2. **Record that commit in the experiment.** Every experiment then names the code that produced it, and `git checkout <commit>` brings that version back:

   ```python
   import subprocess
   commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
   C.start(client, "cookbook-fix-3", description="...", metadata={"commit": commit, "model": A.MODEL})
   ```
3. **Put a change you will compare often behind a switch.** Both versions then live in the code at once and either can run at any time. 05's request cut is one: `agent(..., context_tokens=None)` runs the agent without it.

## 10 · The agent answers, judged

Points made, for the agent answers of all three rounds: the only answers a fix changed, and 18 judge requests, within one day's allowance. Saved one by one; if the limit is reached, run the cell again after the reset and it carries on.

In [3]:
for label, name in rounds:
    verdicts, finished = C.judge(name, kinds=("agent",))
    if finished:
        C.publish(client, name, kinds=("agent",))
for q in [q for q in E.QUESTIONS if q["kind"] == "agent"]:
    print(q["question"][:90])
    for label, name in rounds:
        v = C.saved(name, "verdicts").get(q["question"])
        if v is None:
            print(f"   {label:9} not judged yet")
        elif v.get("no answer"):
            print(f"   {label:9} no answer")
        else:
            s = J.scores(v, q)
            print(f"   {label:9} points made {s['points made']:.0%}   (the judge's leads to check: {s['claims wrong']} claims it marks wrong)")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


    judged 1 of 3: How does chapter 2's runtime stop an agent that keeps callin
    judged 2 of 3: In chapter 8's LangGraph project, compare how the LangGraph 


    judged 3 of 3: How is chapter 6's shared board different from chapter 4's m


View the evaluation results for experiment: 'cookbook-baseline' at:
https://smith.langchain.com/o/0fdf5d9d-7d7b-4da3-9ce3-ece5ff4b3dde/datasets/6d079e56-9d23-4f6b-acbf-c5b786c833cc/compare?selectedSessions=1fab8080-589c-45a9-bf93-8132ac26c169




...

    judged 3 of 3: How is chapter 6's shared board different from chapter 4's m


View the evaluation results for experiment: 'cookbook-fix-2' at:
https://smith.langchain.com/o/0fdf5d9d-7d7b-4da3-9ce3-ece5ff4b3dde/datasets/6d079e56-9d23-4f6b-acbf-c5b786c833cc/compare?selectedSessions=182642b9-a7af-4f89-90df-14aaa332a2f9




0it [00:00, ?it/s]

3it [00:00, 295.15it/s]

How is chapter 6's shared board different from chapter 4's memory stores? When would you u
   baseline  points made 100%   (the judge's leads to check: 2 claims it marks wrong)
   round 2   points made 100%   (the judge's leads to check: 0 claims it marks wrong)
   round 3   points made 100%   (the judge's leads to check: 4 claims it marks wrong)
In chapter 8's LangGraph project, compare how the LangGraph desk and the from-scratch case
   baseline  no answer
   round 2   no answer
   round 3   points made 71%   (the judge's leads to check: 1 claims it marks wrong)
How does chapter 2's runtime stop an agent that keeps calling the same tool, and how does 
   baseline  points made 57%   (the judge's leads to check: 1 claims it marks wrong)
   round 2   points made 71%   (the judge's leads to check: 0 claims it marks wrong)
   round 3   points made 57%   (the judge's leads to check: 0 claims it marks wrong)


## 11 · What LangSmith does that you have to know

All offline.

**LlamaIndex's OpenTelemetry steps vanish inside an experiment.** The client and the server disagree on how a run's id becomes a span id:

In [5]:
show(K.otel_ids())

{
 "the row's run id": "01a11ad7-b618-71b3-b1ab-d927d8bc4db1",
 "span id its children are given": "0x1a11ad7b61871b3",
 "the run id the server makes of it": "00000000-0000-0000-01a1-1ad7b61871b3",
 "the same run": false
}


**`traceable` changes a tool's schema.** What the model would be offered, traced three ways:

In [6]:
show(K.traceable_signature())

{
 "plain": {
  "signature": "(query: str) -> str",
  "the model is offered": [
   "query"
  ]
 },
 "traceable": {
  "signature": "(query: str, *, config=None) -> str",
  "the model is offered": [
   "config",
   "query"
  ]
 },
 "tracing.traced_tool": {
  "signature": "(query: str) -> str",
  "the model is offered": [
   "query"
  ]
 }
}


## Record of the runs

Everything that spent a token, in order. Each answer and verdict is saved in `store/` as it arrived, and the experiments are in LangSmith.

| # | What | Model | Cost | Result |
|---|---|---|---|---|
| 1 | Judge design: 2 of 05's answers marked by hand, one-call judge | gemini-3.5-flash | 2 requests | missed a wrong claim; rewrote the answer's claims into the key's shape |
| 2 | The same 2 answers, two-call judge | gemini-3.5-flash | 4 requests | all three wrong claims found; this design kept |
| 3 | Baseline: 10 questions through 05 as built | gpt-oss-120b | ~63k tokens | 9 answered; board cited no file; LangGraph lost to a refused reply (400) |
| 4 | Baseline judged, for tuning | gemini-3.5-flash | ~16 requests | 8 of 10, then the daily limit; two habits found, check step tightened |
| 5 | Judge measured on the held-out answers | gemini-2.5-flash | 20 requests | points made agree; 2 of 5 wrong claims caught, 7 false alarms |
| 6 | Round 2: fixes 1 and 2 | gpt-oss-120b | ~65k tokens | board cites its files; LangGraph over the size limit (413) |
| 7 | Other judges tried on the held-out answers | gemini-3.8-flash, qwen3.8-27b | 3 answers (qwen) | overloaded; qwen refused after 3 on its output limit |
| 8 | | gpt-oss-20b | 21 requests | 0 of 5 wrong claims caught; one answer with no claims found |
| 9 | Round 3: fixes 3 and 4, agent questions | gpt-oss-120b | ~75k tokens | all three answered and cited; LangGraph answered, right |
| 10 | Agent answers of all three rounds judged | gemini-2.5-flash | 16 requests | points made in section 10; 8 leads, 2–3 real |

**What changed in 05**, and nowhere else:

- `open_file` labels what it returns with its file and lines (fix 1).
- `chat_model` sends a request again, up to twice, when Groq cannot parse the reply (fix 2).
- Each agent request is cut to 6,000 tokens of messages, oldest tool results first (`fit`, fix 3).
- The step-limit answer is given what the run found (`_finishing`, fix 4).
- `chat_model`, `tool_functions`, `tools` and `agent` take an optional `wrap`, where the tracer goes. Without it, 05 runs exactly as before.

## Findings

**The loop worked: three rounds, four fixes, and the question that failed twice answered.** The baseline answered 9 of 10 questions. The board answer cited no usable file, and the LangGraph question was lost to a reply Groq refused. Each fix came from a trace:

- `open_file` now names the file it returns, and the board answer cites its files again.
- A reply Groq cannot parse is asked again. Round 2 had no such reply, so this fix was not tested there; 08's runs used it.
- Each request is cut to 6,000 tokens. Requests now level off near 6,000 instead of growing until one is refused.
- The answer at the step limit is given what the run found. As LlamaIndex ships, that request carries none of it.

Round 3 answered the LangGraph question: by hand, no claim wrong and 5 of its 7 points; by the judge, 71%, which is the same. Each round ran once, so each result is one sample.

**A fix has a cost.** Round 3's LangGraph answer was the most expensive answer in any round: 42,675 tokens, for eight tool calls and the answer after them. The cut caps each request, not the run.

**Points made held steady where nothing was broken.** The board answers made all 6 points in every round. The runtime question went 57%, 71%, 57%: a swing of one point, which one run a round cannot tell apart from chance.

**The judge is good at coverage, not at errors.** This followed the usual procedure: a hand-marked set, tuned on one part, measured on the part held out. On the 10 held-out answers, its "points made" matched the hand marks; it caught 2 of 5 wrong claims and raised 7 false alarms. On the three rounds' answers it flagged 8 claims:

- 2 or 3 were real but small, and the hand check had let them pass. The write tool checks "read-access" (it checks only ownership), and reading the board is "tool-driven" (reading is injection).
- 5 were false. It split one list ("SQLite, Chroma, Neo4j") into three claims, misread a citation, and marked as wrong a claim that the evidence lines it was shown did not reach.

Read its lists, then; don't count them.

**Free tiers shaped everything, starting with the test set's size.** Ten questions, three of them for the agent, each run once a round, is what the free tiers allow: one round is most of a day's Groq tokens, and judging it is a day of the judge's requests. So every difference here is one sample, and the swings on the runtime question are as likely chance as change. Gemini Flash allows about 20 requests a day per model, and stopped offering the checked judge to new keys. The newest Flash could not be reached at all, qwen's output limit is about one claim list a minute, and gpt-oss-20b was too weak. Groq allows 200,000 tokens a day and 8,000 per request. All of it cost cents; what was scarce was requests. A paid key would change which judge is used.

**What LangSmith gave, and what it took.** It gave the loop itself:

- a versioned dataset,
- one experiment per round, compared side by side,
- free checks scored as each answer arrived,
- the judge's verdicts attached afterwards,
- every trace readable back through the API, which is where each fix came from.

It took hand work to trace LlamaIndex. Its OpenTelemetry route loses every LlamaIndex step inside an experiment, because client and server disagree on span ids. Its `traceable` changes a tool's schema unless it is wrapped. And `list_runs`, used here to read traces back, is deprecated (until January 2027).